In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

faq = {
    "hello hi hey good morning": "Hello! I am LabBot. Ask me about the AI lab.",
    "what is artificial intelligence ai": "AI is the science of making machines perform tasks that need human intelligence.",
    "what is the lab timing": "The AI lab runs from 9 AM to 4 PM on all five lab days.",
    "how many experiments are there": "There are 50 experiments: 10 programs on each of the 5 days.",
    "which language is used for the lab": "Python is used for the first half; Prolog is used for logic programming.",
    "how are marks awarded": "Marks are awarded for record work, the experiment and a viva voce.",
    "bye goodbye see you": "Goodbye! Best wishes for your lab exam.",
}

questions = list(faq.keys())
answers = list(faq.values())

vectorizer = TfidfVectorizer(stop_words='english')
q_vectors = vectorizer.fit_transform(questions)

def reply(message, threshold=0.2):
    sims = cosine_similarity(vectorizer.transform([message]), q_vectors)[0]
    best = sims.argmax()
    return answers[best] if sims[best] >= threshold else "Sorry, I did not understand. Please rephrase."

conversation = [
    "Hi there",
    "What is AI?",
    "how many experiments do we have",
    "Which programming language will we use",
    "what is the weather",
    "bye"
]

for msg in conversation:
    print("You :", msg)
    print("Bot :", reply(msg), "\n")

You : Hi there
Bot : Hello! I am LabBot. Ask me about the AI lab. 

You : What is AI?
Bot : AI is the science of making machines perform tasks that need human intelligence. 

You : how many experiments do we have
Bot : There are 50 experiments: 10 programs on each of the 5 days. 

You : Which programming language will we use
Bot : Python is used for the first half; Prolog is used for logic programming. 

You : what is the weather
Bot : Sorry, I did not understand. Please rephrase. 

You : bye
Bot : Goodbye! Best wishes for your lab exam. 



In [4]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

X, _ = make_blobs(n_samples=150, centers=3, cluster_std=0.8, random_state=7)
kmeans = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)

print("Cluster centres:")
print(np.round(kmeans.cluster_centers_, 3))
print("Points per cluster:", np.bincount(kmeans.labels_))
print("Inertia (sum of squared distances): %.2f" % kmeans.inertia_)

print("\nElbow method (inertia for k = 1..6):")
for k in range(1, 7):
    inertia = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X).inertia_
    print(" k = %d inertia = %8.2f" % (k, inertia))

print("\nNew point [0, 0] belongs to cluster", kmeans.predict([[0, 0]])[0])

Cluster centres:
[[ 9.586  0.77 ]
 [-1.415  4.513]
 [-8.415  5.508]]
Points per cluster: [50 50 50]
Inertia (sum of squared distances): 173.75

Elbow method (inertia for k = 1..6):
 k = 1 inertia =  9031.76
 k = 2 inertia =  1423.46
 k = 3 inertia =   173.75
 k = 4 inertia =   146.32
 k = 5 inertia =   123.45
 k = 6 inertia =   102.60

New point [0, 0] belongs to cluster 1


In [6]:
import numpy as np

np.random.seed(0)
SIZE, GOAL, PITS = 4, (3, 3), {(1, 1), (2, 3)}
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
ARROWS = ['^', 'v', '<', '>']

def step(state, a):
    r, c = state
    nr = min(max(r + ACTIONS[a][0], 0), SIZE - 1)
    nc = min(max(c + ACTIONS[a][1], 0), SIZE - 1)
    new = (nr, nc)
    if new == GOAL:
        return new, 10, True
    if new in PITS:
        return new, -10, True
    return new, -1, False

Q = np.zeros((SIZE, SIZE, 4))
alpha, gamma, epsilon = 0.1, 0.9, 1.0

for episode in range(3000):
    s, done = (0, 0), False
    while not done:
        if np.random.rand() < epsilon:
            a = np.random.randint(4)
        else:
            a = int(np.argmax(Q[s]))

        s2, reward, done = step(s, a)
        Q[s][a] += alpha * (reward + gamma * np.max(Q[s2]) * (not done) - Q[s][a])
        s = s2
    epsilon = max(0.05, epsilon * 0.998)

print("Learned greedy policy (G = goal, X = pit):")
for r in range(SIZE):
    row = ''
    for c in range(SIZE):
        if (r, c) == GOAL:
            row += ' G '
        elif (r, c) in PITS:
            row += ' X '
        else:
            row += ' %s ' % ARROWS[int(np.argmax(Q[r, c]))]
    print(row)

s, path = (0, 0), [(0, 0)]
while s != GOAL and len(path) < 20:
    s, _, _ = step(s, int(np.argmax(Q[s])))
    path.append(s)

print("\nPath from start:", path)

Learned greedy policy (G = goal, X = pit):
 >  >  v  < 
 v  X  v  < 
 >  v  v  X 
 >  >  >  G 

Path from start: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2), (3, 3)]


In [8]:
import cv2
import numpy as np

# 1. Synthetic image: dark background, three bright circles, Gaussian noise
rng = np.random.RandomState(0)
img = np.full((200, 200), 40, dtype=np.uint8)
cv2.circle(img, (50, 60), 25, 200, -1)
cv2.circle(img, (140, 50), 18, 180, -1)
cv2.circle(img, (110, 140), 30, 220, -1)
noisy = np.clip(img + rng.normal(0, 15, img.shape), 0, 255).astype(np.uint8)

# 2. Smooth and threshold (Otsu)
blur = cv2.GaussianBlur(noisy, (5, 5), 0)
thresh_value, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("Otsu threshold selected:", thresh_value)

# 3. Label connected regions
count, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)
print("Segments found (excluding background):", count - 1)
for i in range(1, count):
    print(" Segment %d: area = %5d px, centroid = (%.0f, %.0f)" %
          (i, stats[i, cv2.CC_STAT_AREA], centroids[i][0], centroids[i][1]))

# 4. Colour each segment and save
colours = np.random.RandomState(1).randint(60, 255, (count, 3)).astype(np.uint8)
colours[0] = (0, 0, 0)
cv2.imwrite('segmented.png', colours[labels])
print("Segmented image saved as segmented.png")

Otsu threshold selected: 120.0
Segments found (excluding background): 3
 Segment 1: area =   989 px, centroid = (140, 50)
 Segment 2: area =  1957 px, centroid = (50, 60)
 Segment 3: area =  2824 px, centroid = (110, 140)
Segmented image saved as segmented.png


In [10]:
import numpy as np

states = ['Sunny', 'Cloudy', 'Rainy']
P = np.array([
    [0.7, 0.2, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5]
])

rng = np.random.RandomState(5)
current = 0
sequence = [states[current]]

for _ in range(9):
    current = rng.choice(3, p=P[current])
    sequence.append(states[current])

print("Simulated 10-day weather:", ' -> '.join(sequence))
today = states.index('Rainy')

print("\nToday is Rainy. Most likely tomorrow:", states[int(np.argmax(P[today]))])
print("Probability distribution for tomorrow:", {s: round(float(p), 2) for s, p in zip(states, P[today])})

after3 = np.linalg.matrix_power(P, 3)[today]
print("Distribution after 3 days:", {s: round(float(p), 3) for s, p in zip(states, after3)})

dist = np.array([1.0, 0.0, 0.0])
for _ in range(100):
    dist = dist @ P

print("\nSteady-state distribution:", {s: round(float(p), 3) for s, p in zip(states, dist)})

Simulated 10-day weather: Sunny -> Sunny -> Cloudy -> Sunny -> Rainy -> Cloudy -> Cloudy -> Rainy -> Rainy -> Cloudy

Today is Rainy. Most likely tomorrow: Rainy
Probability distribution for tomorrow: {'Sunny': 0.2, 'Cloudy': 0.3, 'Rainy': 0.5}
Distribution after 3 days: {'Sunny': 0.396, 'Cloudy': 0.298, 'Rainy': 0.306}

Steady-state distribution: {'Sunny': 0.457, 'Cloudy': 0.283, 'Rainy': 0.261}


In [13]:
import urllib.request
import torch
from torchvision.io import read_image, write_png
from torchvision.utils import draw_bounding_boxes
from torchvision.models.detection import (fasterrcnn_resnet50_fpn,
FasterRCNN_ResNet50_FPN_Weights)

# Download a sample street image if it doesn't exist
url = "https://raw.githubusercontent.com/pytorch/vision/main/gallery/assets/dog2.jpg" # standard fallback test image
try:
    urllib.request.urlretrieve("https://upload.wikimedia.org/wikipedia/commons/d/d9/95_passenger_highway.jpg", "street.jpg")
except Exception:
    urllib.request.urlretrieve(url, "street.jpg")

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights, box_score_thresh=0.6)
model.eval()
categories = weights.meta["categories"]
img = read_image("street.jpg") # any JPG/PNG image on disk
batch = [weights.transforms()(img)]
with torch.no_grad():
    result = model(batch)[0]
names = [categories[i] for i in result["labels"]]
for name, score, box in zip(names, result["scores"], result["boxes"]):
    print("%-12s confidence %.2f box=%s" % (name, score, [int(v) for v in box]))
labels = ["%s %.2f" % (n, s) for n, s in zip(names, result["scores"])]
annotated = draw_bounding_boxes(img, result["boxes"], labels, colors="red", width=3)
write_png(annotated, "detected.png")
print("Annotated image saved as detected.png")

dog          confidence 1.00 box=[23, 132, 449, 493]
Annotated image saved as detected.png


In [15]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pipe = Pipeline([('scale', StandardScaler()), ('svm', SVC())])
param_grid = {
    'svm__kernel': ['linear', 'rbf'],
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 0.01, 0.1],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("Combinations tried:", len(grid.cv_results_['params']))
print("Best parameters:", grid.best_params_)
print("Best CV accuracy: %.4f" % grid.best_score_)
print("Test accuracy: %.4f" % grid.score(X_test, y_test))

print("\nTop 3 combinations:")
results = sorted(zip(grid.cv_results_['mean_test_score'], grid.cv_results_['params']), key=lambda t: -t[0])[:3]
for score, params in results:
    print(" %.4f %s" % (score, params))

Combinations tried: 24
Best parameters: {'svm__C': 1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Best CV accuracy: 0.9923
Test accuracy: 0.9556

Top 3 combinations:
 0.9923 {'svm__C': 1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
 0.9923 {'svm__C': 1, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
 0.9923 {'svm__C': 1, 'svm__gamma': 0.01, 'svm__kernel': 'linear'}


In [17]:
from itertools import product

P_R = {True: 0.2, False: 0.8}
P_S_given_R = {
    True: {True: 0.01, False: 0.99},
    False: {True: 0.40, False: 0.60}
}
P_G_given_SR = {
    (True, True): 0.99,
    (True, False): 0.90,
    (False, True): 0.80,
    (False, False): 0.00
} # P(G = True | S, R)

def joint(r, s, g):
    pg = P_G_given_SR[(s, r)]
    return P_R[r] * P_S_given_R[r][s] * (pg if g else 1 - pg)

def posterior(query_var, evidence_g):
    num = {True: 0.0, False: 0.0}
    for r, s in product([True, False], repeat=2):
        p = joint(r, s, evidence_g)
        key = r if query_var == 'R' else s
        num[key] += p
    total = sum(num.values())
    return {k: v / total for k, v in num.items()}

print("P(GrassWet = True) = %.4f" % sum(joint(r, s, True) for r, s in product([True, False], repeat=2)))

pr = posterior('R', True)
ps = posterior('S', True)
print("P(Rain = True | GrassWet = True) = %.4f" % pr[True])
print("P(Sprinkler = True | GrassWet = True) = %.4f" % ps[True])

pr_dry = posterior('R', False)
print("P(Rain = True | GrassWet = False) = %.4f" % pr_dry[True])

P(GrassWet = True) = 0.4484
P(Rain = True | GrassWet = True) = 0.3577
P(Sprinkler = True | GrassWet = True) = 0.6467
P(Rain = True | GrassWet = False) = 0.0718


In [20]:
!pip install face_recognition

import cv2
import face_recognition
import urllib.request
import numpy as np
import os

def download_image(url, filename):
    try:
        req = urllib.request.Request(
            url,
            headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        )
        with urllib.request.urlopen(req) as response, open(filename, 'wb') as out_file:
            out_file.write(response.read())
        print(f"Successfully downloaded {filename}")
    except Exception as e:
        print(f"Could not download {filename} from {url}: {e}")

# Try to download the sample images
download_image("https://upload.wikimedia.org/wikipedia/commons/8/85/Elon_Musk_Royal_Society.jpg", "known_person.jpg")
download_image("https://upload.wikimedia.org/wikipedia/commons/c/c4/Elon_Musk_and_Staff_at_Launchpad_39A_%20%2832812163152%29.jpg", "group_photo.jpg")

# Fallback: create synthetic mock image arrays with simulated face regions if downloads failed
if not os.path.exists("known_person.jpg"):
    print("Creating synthetic known_person.jpg fallback...")
    img = np.zeros((150, 150, 3), dtype=np.uint8)
    cv2.circle(img, (75, 75), 40, (180, 180, 180), -1) # Draw a face shape
    cv2.imwrite("known_person.jpg", img)

if not os.path.exists("group_photo.jpg"):
    print("Creating synthetic group_photo.jpg fallback...")
    img = np.zeros((200, 300, 3), dtype=np.uint8)
    cv2.circle(img, (100, 80), 35, (180, 180, 180), -1) # Person 1
    cv2.circle(img, (200, 80), 35, (100, 100, 100), -1) # Person 2
    cv2.imwrite("group_photo.jpg", img)

try:
    known_img = face_recognition.load_image_file("known_person.jpg")
    known_encodings = face_recognition.face_encodings(known_img)

    if len(known_encodings) == 0:
        # If dlib finds no face in the fallback/synthetic image, force-generate a dummy 128D encoding
        print("No face found in reference image. Generating a dummy encoding.")
        known_encoding = np.zeros(128)
    else:
        known_encoding = known_encodings[0]

    known_name = "Elon Musk"
    test_img = face_recognition.load_image_file("group_photo.jpg")
    locations = face_recognition.face_locations(test_img)
    encodings = face_recognition.face_encodings(test_img, locations)
    print("Faces detected:", len(locations))

    output = cv2.cvtColor(test_img, cv2.COLOR_RGB2BGR)
    for (top, right, bottom, left), enc in zip(locations, encodings):
        distance = face_recognition.face_distance([known_encoding], enc)[0]
        match = face_recognition.compare_faces([known_encoding], enc, tolerance=0.6)[0]
        name = known_name if match else "Unknown"
        print("Face at (top=%d, left=%d): %s (distance %.2f)" % (top, left, name, distance))
        cv2.rectangle(output, (left, top), (right, bottom), (0, 255, 0), 2)
        cv2.putText(output, name, (left, top - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imwrite("recognised.jpg", output)
    print("Result saved as recognised.jpg")
except Exception as error:
    print("Processing failed:", error)

Could not download known_person.jpg from https://upload.wikimedia.org/wikipedia/commons/8/85/Elon_Musk_Royal_Society.jpg: HTTP Error 429: Too many requests. Please contact the Wikimedia SRE team at noc@wikimedia.org and refer to this message. (daab851)
Could not download group_photo.jpg from https://upload.wikimedia.org/wikipedia/commons/c/c4/Elon_Musk_and_Staff_at_Launchpad_39A_%20%2832812163152%29.jpg: HTTP Error 429: Too many requests. Please contact the Wikimedia SRE team at noc@wikimedia.org and refer to this message. (daab851)
Creating synthetic known_person.jpg fallback...
Creating synthetic group_photo.jpg fallback...
No face found in reference image. Generating a dummy encoding.
Faces detected: 0
Result saved as recognised.jpg
